# Assignment 11: Production Defense-in-Depth Pipeline

**Course:** AICB-P1 — AI Agent Development  
**Student:** VinBank Security Team  

---

## Context & Architecture

In production systems, no single safety layer is sufficient. This notebook demonstrates a **Defense-in-Depth Pipeline** implementing multiple independent safety layers acting as a series of checks on user inputs and model outputs.

### Pipeline Layers:
1. **Rate Limiter:** Sliding-window rate limiter per user (default 10 requests / 60 seconds) to prevent DDoS and brute-force scanning.
2. **Session Anomaly Detector (Bonus Layer):** Tracks security violations. If a user triggers 3 violations (rate limit, prompt injection, off-topic, etc.), their session is locked for 5 minutes.
3. **Input Guardrails:** Detects prompt injection via regex and filters out-of-scope topics.
4. **Output Guardrails:** Redacts PII and internal secrets (passwords, database URLs, API keys) from the response.
5. **LLM-as-Judge:** Evaluates the safety, relevance, accuracy, and tone of the response on a 1-5 scale using Gemini (with fallback to local heuristics).
6. **Audit Logging & Monitoring:** Tracks latencies, metrics, and triggers alerts for high block rates, session lockouts, and security incidents.

In [ ]:
import os
import sys
import asyncio

# Add the src directory to the python path to import the module
sys.path.append(os.path.abspath('../src'))

# Set a dummy key if GOOGLE_API_KEY is not already set in the environment
if 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = 'dummy_key'
    print("Using dummy GOOGLE_API_KEY. The LLM-as-Judge will use fallback heuristic rules.")
else:
    print("GOOGLE_API_KEY found. LLM-as-Judge will use Gemini API.")

## Initializing and Running the Defense Pipeline

We will run the built-in test suites that execute:
- **Test 1:** Safe Queries (should pass)
- **Test 2:** Attacks (should be blocked/redacted)
- **Test 3:** Rate Limiting & Session Lockouts (rapid queries from a single user)
- **Test 4:** Edge cases (empty inputs, SQL injections, off-topic queries)

In [ ]:
from assignment11_defense_pipeline import run_assignment_tests

# Run the full test suite asynchronously
await run_assignment_tests()

## Inspecting the Exported Audit Logs

The pipeline exports detailed interaction logs to `security_audit.json`. Let's inspect the first few logs to verify the metadata collection (latency, block status, security layers matched).

In [ ]:
import json

with open('security_audit.json', 'r', encoding='utf-8') as f:
    logs = json.load(f)

print(f"Total logged interactions: {len(logs)}\n")
print("First 10 logged interactions:")
for i, entry in enumerate(logs[:10], 1):
    print(f"\nInteraction #{i}:")
    print(f"  Timestamp:  {entry['timestamp']}")
    print(f"  User ID:    {entry['user_id']}")
    print(f"  Input:      {entry['input']}")
    print(f"  Status:     {entry['status']}")
    print(f"  Blocked By: {entry['blocked_by']}")
    print(f"  Latency:    {entry['latency_ms']:.2f} ms")
    if entry['scores']:
        print(f"  Judge Scores: {entry['scores']}")